# Concept 4 (Advanced): Validation & Retry Loops

Asks the model for raw JSON as plain text, which is more failure-prone than a tool call, then validates and retries if needed.

## Setup
Store your key in Colab **Secrets** (key icon, left sidebar) as `OPENAI_API_KEY`.

In [ ]:
!pip install -q openai

In [ ]:
from openai import OpenAI
from google.colab import userdata
client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

In [ ]:
import json
from jsonschema import validate, ValidationError
schema = {"type": "object", "properties": {"city": {"type": "string"}, "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}}, "required": ["city", "unit"]}
def call_model(prompt, error_feedback=None):
    messages = [{"role": "system", "content": "Extract city and unit. Respond with ONLY raw JSON, no markdown, matching: " + json.dumps(schema)}, {"role": "user", "content": prompt}]
    if error_feedback:
        messages.append({"role": "user", "content": error_feedback})
    reply = client.chat.completions.create(model="gpt-4o-mini", messages=messages, temperature=0.7)
    return reply.choices[0].message.content
def get_valid_arguments(prompt, max_retries=3):
    error_feedback = None
    for attempt in range(1, max_retries + 1):
        raw = call_model(prompt, error_feedback)
        try:
            args = json.loads(raw)
            validate(instance=args, schema=schema)
            return args, attempt
        except (json.JSONDecodeError, ValidationError) as e:
            print(f"Attempt {attempt} failed: {raw!r}")
            error_feedback = f"Invalid: {e}. Return ONLY raw JSON, no markdown."
    raise RuntimeError("Failed after retries")
args, attempts = get_valid_arguments("I'm travelling to Pune tomorrow, what should I pack? Use celsius.")
print(f"Succeeded on attempt {attempts}: {args}")

## How to Extend
Run the cell a few times and watch most attempts succeed on try 1, with an occasional retry. Set `max_retries=1` and re-run until you see it fail on purpose.